## Import the libraries


In [10]:
%shell uv sync

Resolved 128 packages in 2ms
Checked 120 packages in 2ms


In [13]:
import sys
import os

# Añadir el entorno virtual de uv al sys.path para que detecte las dependencias instaladas
# On Google Collab
venv_path = "/content/.venv/lib/python3.13/site-packages"
if os.path.exists(venv_path) and venv_path not in sys.path:
    sys.path.insert(0, venv_path)

from collections import Counter
import json
import re

import torch
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import snapshot_download
from peft import LoraConfig, prepare_model_for_kbit_training

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

## Check the GPU


In [14]:
if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required for this project")

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


Tesla T4
Memory: 14.6 GB


## Create the project folders


In [1]:
WORKSPACE = "/tmp/tensortonic"
DATA_DIR = os.path.join(WORKSPACE, "data", "medquad")
MODEL_DIR = os.path.join(WORKSPACE, "models")
MODEL_CACHE = os.path.join(MODEL_DIR, "huggingface")
DATASET_CACHE = os.path.join(WORKSPACE, "cache", "datasets")

os.makedirs(MODEL_CACHE, exist_ok=True)
os.makedirs(DATASET_CACHE, exist_ok=True)

print(WORKSPACE)
print(MODEL_DIR)
print(MODEL_CACHE)
print(DATASET_CACHE)


NameError: name 'os' is not defined

## Download the base model


In [16]:
MODEL_ID = "Qwen/Qwen3-4B"
print(f"Downloading {MODEL_ID} to {MODEL_CACHE}", flush=True)
local_model_path = snapshot_download(MODEL_ID, cache_dir=MODEL_CACHE)
print(f"Model files ready at {local_model_path}", flush=True)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Model files ready at /tmp/tensortonic/models/huggingface/models--Qwen--Qwen3-4B/snapshots/1cfa9a7208912126459214e8b04321603b3df60c


## Configure 4-bit quantization


In [18]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    # TODO 1: configure NF4, bfloat16 compute, and double quantization.
    bnb_4bit_quant_type="nf4", #nf4 is design to adapt normal distribution gettin less losing that standard 4 bits
    bnb_4bit_compute_dtype=torch.bfloat16, # calculating matrix on bfloat16 is more optimal
    bnb_4bit_use_double_quant=True,
)
print(quantization)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## Load the tokenizer and model


Tokenizer and pad strategy

loading and quantization on GPU

In [19]:
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LOAD AND FREEZE QUANTIZED BASE LLM ON GPU 4 APPLY ADAPTERS QLORA
base_model = AutoModelForCausalLM.from_pretrained(
    local_model_path, # read blobs from llm located on ssd
    quantization_config=quantization, # applied previous quantization config
    device_map={"": 0}, #set all quantized model on number 0 gpu instead of ram cpu
    torch_dtype=torch.bfloat16, #define precision 4 intermediate layers
)
base_model.config.use_cache = False # cache on trainin is useless, it must be used on inference
base_model = prepare_model_for_kbit_training(base_model) # 1. freeze llm base wights knowledge. 2. keep critic layers on fp32 3. prepare 4 compute gradients on trainin
print(f"Loaded {MODEL_ID}")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B


## Download MedQuAD dataset 4 trainin


Medical Question Answering Dataset. Es un conjunto de datos muy famoso en medicina que contiene miles de parejas de preguntas y respuestas sobre salud, enfermedades y medicamentos, extraídas de fuentes médicas oficiales (como los Institutos Nacionales de Salud de EE.UU., NIH)

In [20]:
DATASET_ID = "Hmehdi515/MedQuad"
DATASET_FILES = {
    "train": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_train.csv",
    "validation": "https://huggingface.co/datasets/Hmehdi515/MedQuad/resolve/main/medquad_val.csv",
}

print(f"Downloading {DATASET_ID}", flush=True)
#download remote csv 4 trainin and save em to cache
source = load_dataset("csv", data_files=DATASET_FILES, cache_dir=DATASET_CACHE)

#convert csv to jsonl and save it
os.makedirs(DATA_DIR, exist_ok=True)
raw_train_path = os.path.join(DATA_DIR, "raw_train.jsonl")
raw_validation_path = os.path.join(DATA_DIR, "raw_validation.jsonl")
source["train"].to_json(raw_train_path) # llm trainin standard practice. convert to jsonl
source["validation"].to_json(raw_validation_path)

print(f"Saved {len(source['train'])} rows to {raw_train_path}", flush=True)
print(f"Saved {len(source['validation'])} rows to {raw_validation_path}", flush=True)
print(source)


medquad_train.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

medquad_train.csv: downloading bytes:           |  0.00B            

medquad_val.csv:   0%|          | 0.00/199k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Creating json from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved 16078 rows to /tmp/tensortonic/data/medquad/raw_train.jsonl
Saved 164 rows to /tmp/tensortonic/data/medquad/raw_validation.jsonl
DatasetDict({
    train: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 16078
    })
    validation: Dataset({
        features: ['sample', 'question', 'answer'],
        num_rows: 164
    })
})


## Inspect the dataset


Inspect the available columns and calculate the median answer length for the training split.

Check: The available dataset columns, a non-empty list of answer lengths, and one median value.

Read column_names from the training split. Sort the word counts. For an even number of answers, average the two middle values; otherwise select the middle value.

In [ ]:
# TODO 2: inspect the columns and calculate the median answer length in words.
columns = []
answer_lengths = []
median_answer_words = None

assert {"question", "answer"}.issubset(columns), "Check the dataset columns"
assert answer_lengths, "Calculate the answer lengths"
assert median_answer_words is not None, "Calculate the median answer length"

print({
    "columns": columns,
    "training_rows": len(source["train"]),
    "median_answer_words": median_answer_words,
})
print(source["train"][0])


## Prepare a clean training split
